# Chinook SQL Agent 演示

离线回放验证流程，不代表大模型成绩。请先在项目根目录运行测试。

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd() if (Path.cwd() / "sql_agent").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from sql_agent.catalog import parse_database
from sql_agent.model import ReplayModel
from sql_agent.workflow import Workflow
CATALOG = parse_database(ROOT / "data/chinook.db", ROOT / "artifacts/catalog")
{n: t["row_count"] for n,t in CATALOG["tables"].items()}

## 正常路径
从显式回放fixture读取节点输出，实际执行SQL。

In [ ]:
cases = json.loads((ROOT / "evaluation/datasets/seed.json").read_text(encoding="utf-8"))
fixtures = json.loads((ROOT / "evaluation/replay_fixtures.json").read_text(encoding="utf-8"))
def replay(case_id):
    case = next(c for c in cases if c["id"] == case_id)
    flow = Workflow(ROOT / "data/chinook.db", ReplayModel(fixtures[case_id]), ROOT / "config/business_rules.json", ROOT / "artifacts")
    return flow.run(case["question"], case["reference_date"])
state = replay("S11")
state.status, state.execution.rows

## 人为注入错误后的自动修复
第一稿引用不存在的字段，第二稿修复。不能把这个案例描述为真实模型的自然错误。

In [ ]:
fixture = json.loads((ROOT / "examples/repair_fixture.json").read_text(encoding="utf-8"))
flow = Workflow(ROOT / "data/chinook.db", ReplayModel(fixture), ROOT / "config/business_rules.json", ROOT / "artifacts")
state = flow.run("客户总人数是多少？", "2014-01-15")
[(e["node"], e["status"]) for e in state.trace]

## 澄清、不支持与无记录
检查状态及解释，不自动变更问题条件。

In [ ]:
[(case_id, replay(case_id).status) for case_id in ["C01", "U01", "S26"]]

## 真实模型
在启动Notebook的终端配置SQL_AGENT_BASE_URL、SQL_AGENT_API_KEY、SQL_AGENT_MODEL。下面代码仅展示调用方式，执行会产生API用量。

In [ ]:
# from sql_agent.model import OpenAICompatibleModel
# model = OpenAICompatibleModel(ROOT / "config/model_example.json")
# flow = Workflow(ROOT / "data/chinook.db", model, ROOT / "config/business_rules.json", ROOT / "artifacts")
# state = flow.run("2013年每个有订单月份的销售额是多少？", "2014-01-15")
# print(state.answer.text if state.answer else state.error)

## 评测
命令：python -m sql_agent evaluate --mode replay --split all --report replay_smoke

参考题集尚待人工语义审核。报告中的回放结果仅表示fixture与执行路径一致。